# Acquisition des donnees -- JO 2028

---

## Contexte

Les Jeux Olympiques de Los Angeles 2028 approchent. Ce projet de **data storytelling** vise a
analyser l'histoire olympique a travers les donnees, de 1896 a 2024, pour en tirer des
perspectives sur l'edition 2028.

Ce premier notebook constitue la **phase d'acquisition**. Son objectif est triple :

1. **Inventorier** les sources de donnees disponibles et documenter leur provenance.
2. **Charger** chaque dataset et en dresser un premier portrait statistique.
3. **Verifier** la coherence entre les differentes sources avant de passer au nettoyage.

Aucune transformation n'est effectuee ici : on se contente d'observer les donnees brutes
telles qu'elles sont.

## Imports

In [ ]:
import pandas as pd
import numpy as np
import os

# Configuration d'affichage pour une meilleure lisibilite
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.float_format", "{:.2f}".format)

RAW_DIR = os.path.join("..", "data", "raw")

print("Fichiers disponibles dans data/raw/ :")
for f in sorted(os.listdir(RAW_DIR)):
    size_mb = os.path.getsize(os.path.join(RAW_DIR, f)) / (1024 * 1024)
    print(f"  {f:30s} ({size_mb:.1f} Mo)")

## Presentation des sources

Les donnees proviennent de repositories GitHub publics qui compilent des informations
historiques sur les Jeux Olympiques. Voici le detail de chaque source.

### Source 1 -- KeithGalli/Olympics-Dataset

| Champ       | Detail |
|-------------|--------|
| **URL**     | https://github.com/KeithGalli/Olympics-Dataset |
| **Fichiers** | `bios.csv`, `bios_locs.csv`, `results.csv` |
| **Description** | Dataset complet des athletes olympiques (biographies, localisations, resultats) scrape depuis les sources officielles olympiques. Couvre les JO d'ete et d'hiver de 1896 a 2024. |
| **Lignes attendues** | ~145 500 (bios), ~145 500 (bios_locs), ~308 400 (results) |
| **Licence** | MIT |

### Source 2 -- KeithGalli/Olympics-Dataset/clean-data

| Champ       | Detail |
|-------------|--------|
| **URL**     | https://github.com/KeithGalli/Olympics-Dataset/tree/main/clean-data |
| **Fichiers** | `noc_regions.csv`, `populations.csv` |
| **Description** | Donnees de reference : mapping des codes NOC vers les noms de pays/regions, et populations par pays de 1960 a 2023 (origine : Banque Mondiale). |
| **Lignes attendues** | ~229 (noc_regions), ~266 (populations) |
| **Licence** | MIT |

### Source 3 -- DjangoMustang/Olympics-1896-2024-Tableau

| Champ       | Detail |
|-------------|--------|
| **URL**     | https://github.com/DjangoMustang/Olympics-1896-2024-Tableau |
| **Fichiers** | `olympics_1896_2024.csv`, `city_locations.csv` |
| **Description** | Dataset consolide des medailles olympiques de 1896 a 2024 (incluant Paris 2024), accompagne des coordonnees geographiques des villes hotes. Concu pour la visualisation Tableau. |
| **Lignes attendues** | ~24 000 (olympics_1896_2024), ~55 (city_locations) |
| **Licence** | Libre (GitHub public) |

---

## Chargement des datasets

On charge chaque fichier CSV et on realise un premier examen : dimensions, types de colonnes,
apercu des premieres lignes et statistiques descriptives.

### 1. `bios.csv` -- Biographies des athletes

Ce fichier contient les informations biographiques de tous les athletes olympiques.
Chaque ligne represente un athlete unique identifie par `athlete_id`. On y retrouve
le nom complet, le sexe, le pays (NOC), les dates de naissance/deces, et des
informations complementaires (mensurations, affiliations, surnoms).

In [ ]:
bios = pd.read_csv(os.path.join(RAW_DIR, "bios.csv"))

print(f"Dimensions : {bios.shape[0]:,} lignes x {bios.shape[1]} colonnes")
print(f"\nTypes des colonnes :")
print(bios.dtypes)
print(f"\nApercu des premieres lignes :")
bios.head()

In [ ]:
print("Statistiques descriptives (colonnes numeriques) :")
bios.describe()

In [ ]:
print("Valeurs manquantes par colonne :")
missing = bios.isnull().sum()
missing_pct = (missing / len(bios) * 100).round(1)
pd.DataFrame({"manquantes": missing, "pourcentage": missing_pct}).query("manquantes > 0").sort_values("pourcentage", ascending=False)

**Observations sur `bios.csv` :**

- Le fichier contient environ 145 500 athletes, ce qui couvre l'ensemble de l'histoire olympique.
- Plusieurs colonnes presentent un fort taux de valeurs manquantes (`Died`, `Nick/petnames`, `Title(s)`, `Original name`) -- ce qui est attendu pour des champs optionnels.
- Les colonnes cles pour nos analyses seront : `athlete_id`, `Sex`, `NOC`, `Born`, `Measurements`.
- La colonne `Born` est en format texte (ex: "12 December 1886 in Bordeaux...") et necessiterait un parsing lors du nettoyage.

### 2. `bios_locs.csv` -- Biographies avec geolocalisation

Version structuree des biographies, avec les informations de naissance deja parsees
(ville, region, pays) et les coordonnees geographiques (latitude/longitude).
Contient aussi la taille et le poids extraits de la colonne `Measurements` du fichier precedent.

In [ ]:
bios_locs = pd.read_csv(os.path.join(RAW_DIR, "bios_locs.csv"))

print(f"Dimensions : {bios_locs.shape[0]:,} lignes x {bios_locs.shape[1]} colonnes")
print(f"\nTypes des colonnes :")
print(bios_locs.dtypes)
print(f"\nApercu des premieres lignes :")
bios_locs.head()

In [ ]:
print("Statistiques descriptives :")
bios_locs.describe()

In [ ]:
print("Valeurs manquantes par colonne :")
missing = bios_locs.isnull().sum()
missing_pct = (missing / len(bios_locs) * 100).round(1)
pd.DataFrame({"manquantes": missing, "pourcentage": missing_pct}).query("manquantes > 0").sort_values("pourcentage", ascending=False)

**Observations sur `bios_locs.csv` :**

- Meme nombre de lignes que `bios.csv` : il y a bien une correspondance 1:1 via `athlete_id`.
- Les colonnes `height_cm` et `weight_kg` sont numeriques et pretes a l'emploi, mais presentent beaucoup de valeurs manquantes (mensurations non disponibles pour les athletes historiques).
- Les coordonnees `lat`/`long` permettront des visualisations cartographiques sur l'origine geographique des athletes.
- La colonne `born_date` est au format ISO (YYYY-MM-DD), beaucoup plus exploitable que le champ `Born` de `bios.csv`.

### 3. `results.csv` -- Resultats olympiques

Le coeur du dataset : chaque ligne represente la participation d'un athlete a une epreuve
lors d'une edition des Jeux. On y retrouve le classement (`Pos`), la medaille eventuelle,
la discipline et l'epreuve.

In [ ]:
results = pd.read_csv(os.path.join(RAW_DIR, "results.csv"))

print(f"Dimensions : {results.shape[0]:,} lignes x {results.shape[1]} colonnes")
print(f"\nTypes des colonnes :")
print(results.dtypes)
print(f"\nApercu des premieres lignes :")
results.head()

In [ ]:
print("Statistiques descriptives :")
results.describe(include="all")

In [ ]:
print("Valeurs manquantes par colonne :")
missing = results.isnull().sum()
missing_pct = (missing / len(results) * 100).round(1)
pd.DataFrame({"manquantes": missing, "pourcentage": missing_pct}).query("manquantes > 0").sort_values("pourcentage", ascending=False)

In [ ]:
print("Distribution des medailles :")
print(results["Medal"].value_counts(dropna=False))

print(f"\nNombre d'editions (Games) distinctes : {results['Games'].nunique()}")
print(f"Nombre de disciplines distinctes : {results['Discipline'].nunique()}")

**Observations sur `results.csv` :**

- Plus de 308 000 participations enregistrees, couvrant toutes les editions.
- La colonne `Medal` est majoritairement vide (NaN) : normal, seuls les medailles sont renseignes.
- La colonne `Unnamed: 7` est entierement vide -- artefact d'export a ignorer.
- La colonne `Games` contient le format "YYYY Season Olympics" (ex: "2024 Summer Olympics"), ce qui permettra d'extraire l'annee et la saison.
- Le lien avec les biographies se fait via `athlete_id`.

### 4. `olympics_1896_2024.csv` -- Medailles 1896-2024

Dataset complementaire focalise uniquement sur les medaillees. Plus compact que `results.csv`,
il a l'avantage d'inclure les donnees de Paris 2024 dans un format propre et de contenir
le nom de la ville hote et l'annee directement en colonnes.

In [ ]:
olympics = pd.read_csv(os.path.join(RAW_DIR, "olympics_1896_2024.csv"))

print(f"Dimensions : {olympics.shape[0]:,} lignes x {olympics.shape[1]} colonnes")
print(f"\nTypes des colonnes :")
print(olympics.dtypes)
print(f"\nApercu des premieres lignes :")
olympics.head()

In [ ]:
print("Statistiques descriptives :")
olympics.describe(include="all")

In [ ]:
print("Valeurs manquantes par colonne :")
missing = olympics.isnull().sum()
missing_pct = (missing / len(olympics) * 100).round(1)
pd.DataFrame({"manquantes": missing, "pourcentage": missing_pct}).query("manquantes > 0").sort_values("pourcentage", ascending=False)

In [ ]:
print("Repartition par type de medaille :")
print(olympics["medal_type"].value_counts())

print(f"\nPeriode couverte : {olympics['event_year'].min()} - {olympics['event_year'].max()}")
print(f"Nombre de villes hotes : {olympics['city'].nunique()}")
print(f"\nVilles hotes :")
print(olympics.groupby("city")["event_year"].apply(lambda x: sorted(x.unique())).to_string())

**Observations sur `olympics_1896_2024.csv` :**

- Environ 24 000 medailles enregistrees, de 1896 a 2024.
- Format propre avec `event_year` et `city` directement exploitables.
- Contient les colonnes `event_gender` et `event_type` qui permettent de distinguer les epreuves individuelles des epreuves par equipe.
- Ce dataset sera particulierement utile pour les analyses de tendances sur les medailles.

### 5. `noc_regions.csv` -- Mapping NOC / regions

Table de reference qui associe chaque code NOC (National Olympic Committee) a un nom
de pays ou de region. Indispensable pour les jointures et les visualisations par pays.

In [ ]:
noc_regions = pd.read_csv(os.path.join(RAW_DIR, "noc_regions.csv"))

print(f"Dimensions : {noc_regions.shape[0]:,} lignes x {noc_regions.shape[1]} colonnes")
print(f"\nTypes des colonnes :")
print(noc_regions.dtypes)
print(f"\nApercu :")
noc_regions.head(10)

In [ ]:
print("Valeurs manquantes :")
print(noc_regions.isnull().sum())
print(f"\nNombre de codes NOC uniques : {noc_regions['NOC'].nunique()}")
print(f"Doublons sur NOC : {noc_regions['NOC'].duplicated().sum()}")

**Observations sur `noc_regions.csv` :**

- 229 codes NOC references, ce qui couvre largement les comites olympiques actuels et historiques.
- La colonne `notes` est souvent vide : elle ne contient des precisions que pour les cas particuliers (pays disparus, renommes, etc.).
- Servira de table de jointure pour normaliser les noms de pays a travers les datasets.

### 6. `populations.csv` -- Population par pays (1960-2023)

Donnees de population issues de la Banque Mondiale. Chaque ligne correspond a un pays,
avec une colonne par annee de 1960 a 2023. Permettra de normaliser les performances
olympiques par habitant.

In [ ]:
populations = pd.read_csv(os.path.join(RAW_DIR, "populations.csv"))

print(f"Dimensions : {populations.shape[0]:,} lignes x {populations.shape[1]} colonnes")
print(f"\nColonnes (2 premieres + 5 dernieres annees) :")
print(f"  Identifiants : {list(populations.columns[:2])}")
print(f"  Annees : {list(populations.columns[2:5])} ... {list(populations.columns[-3:])}")
print(f"\nApercu :")
populations.head()

In [ ]:
print(f"Nombre de pays/entites : {len(populations)}")
print(f"Periode couverte : 1960 - 2023")

# Taux de valeurs manquantes sur les dernieres annees
year_cols = [str(y) for y in range(2019, 2024)]
for col in year_cols:
    if col in populations.columns:
        pct = populations[col].isnull().mean() * 100
        print(f"  Manquantes en {col} : {pct:.1f}%")

**Observations sur `populations.csv` :**

- 266 entites (pays + regions agregees comme "Africa Eastern and Southern").
- Le `Country Code` utilise le format ISO 3166-1 alpha-3, qui differe parfois des codes NOC. Une table de correspondance sera necessaire.
- Les populations recentes (2020-2023) sont bien renseignees pour la plupart des pays.

### 7. `city_locations.csv` -- Coordonnees des villes hotes

Petit fichier de reference qui associe chaque edition des JO a la ville hote
et ses coordonnees geographiques. Utile pour les cartes.

In [ ]:
city_locations = pd.read_csv(os.path.join(RAW_DIR, "city_locations.csv"))

print(f"Dimensions : {city_locations.shape[0]:,} lignes x {city_locations.shape[1]} colonnes")
print(f"\nContenu complet :")
city_locations

**Observations sur `city_locations.csv` :**

- 55 entrees couvrant toutes les editions des JO d'ete et d'hiver.
- Certaines villes apparaissent plusieurs fois (Paris : 1900, 1924, 2024 ; Londres : 1908, 1948, 2012).
- Les coordonnees sont en format decimal, directement exploitables pour la cartographie.

---

## Vue d'ensemble

Maintenant que chaque dataset a ete charge individuellement, construisons un tableau
recapitulatif pour avoir une vision globale de notre corpus de donnees.

In [ ]:
datasets = {
    "bios": bios,
    "bios_locs": bios_locs,
    "results": results,
    "olympics_1896_2024": olympics,
    "noc_regions": noc_regions,
    "populations": populations,
    "city_locations": city_locations,
}

summary_rows = []
for name, df in datasets.items():
    n_rows, n_cols = df.shape
    nan_pct = (df.isnull().sum().sum() / (n_rows * n_cols) * 100)
    mem_mb = df.memory_usage(deep=True).sum() / (1024 * 1024)
    summary_rows.append({
        "Dataset": name,
        "Lignes": f"{n_rows:,}",
        "Colonnes": n_cols,
        "NaN (%)": f"{nan_pct:.1f}%",
        "Memoire (Mo)": f"{mem_mb:.1f}",
        "Colonnes (liste)": ", ".join(df.columns[:6].tolist()) + ("..." if n_cols > 6 else ""),
    })

summary_df = pd.DataFrame(summary_rows)
summary_df

Le corpus total represente environ 750 000 lignes reparties sur 7 fichiers. Les deux
fichiers principaux (`bios` et `results`) constituent l'essentiel du volume. Les fichiers
de reference (`noc_regions`, `populations`, `city_locations`) sont legers mais indispensables
pour enrichir les analyses.

---

## Verification de coherence

Avant de passer au nettoyage, verifions que les datasets sont coherents entre eux
et qu'aucune donnee critique ne manque.

### Athletes : `bios` vs `results`

L'`athlete_id` est la cle de jointure entre les biographies et les resultats.
Verifions la correspondance.

In [ ]:
bios_ids = set(bios["athlete_id"].dropna().unique())
results_ids = set(results["athlete_id"].dropna().unique())
bios_locs_ids = set(bios_locs["athlete_id"].dropna().unique())

print(f"Athletes dans bios :        {len(bios_ids):,}")
print(f"Athletes dans bios_locs :    {len(bios_locs_ids):,}")
print(f"Athletes dans results :      {len(results_ids):,}")

print(f"\n-- Intersections --")
print(f"Dans bios ET results :       {len(bios_ids & results_ids):,}")
print(f"Dans bios MAIS PAS results : {len(bios_ids - results_ids):,}")
print(f"Dans results MAIS PAS bios : {len(results_ids - bios_ids):,}")

print(f"\n-- Correspondance bios / bios_locs --")
print(f"Identiques : {bios_ids == bios_locs_ids}")

### Codes NOC : `results` vs `noc_regions`

Verifions que tous les codes NOC presents dans les resultats sont bien references
dans la table de mapping.

In [ ]:
results_nocs = set(results["NOC"].dropna().unique())
ref_nocs = set(noc_regions["NOC"].dropna().unique())

missing_nocs = results_nocs - ref_nocs

print(f"Codes NOC dans results :     {len(results_nocs)}")
print(f"Codes NOC dans noc_regions : {len(ref_nocs)}")
print(f"\nNOC dans results MAIS PAS dans noc_regions ({len(missing_nocs)}) :")

if missing_nocs:
    for noc in sorted(missing_nocs):
        count = results[results["NOC"] == noc].shape[0]
        print(f"  {noc} ({count} participations)")
else:
    print("  Aucun -- tous les NOC sont references.")

### Couverture temporelle

Verifions les periodes couvertes par chaque source et assurons-nous que
les donnees les plus recentes (Paris 2024) sont bien presentes.

In [ ]:
# Extraction de l'annee depuis la colonne Games de results.csv
results["year"] = results["Games"].str.extract(r"(\d{4})").astype(float)

print("=== Couverture temporelle ===")
print(f"\nresults.csv :")
print(f"  Annees : {int(results['year'].min())} - {int(results['year'].max())}")
print(f"  Editions : {results['Games'].nunique()}")

print(f"\nolympics_1896_2024.csv :")
print(f"  Annees : {olympics['event_year'].min()} - {olympics['event_year'].max()}")

print(f"\ncity_locations.csv :")
print(f"  Annees : {city_locations['Year'].min()} - {city_locations['Year'].max()}")

print(f"\npopulations.csv :")
year_cols_pop = [c for c in populations.columns if c.isdigit()]
print(f"  Annees : {min(year_cols_pop)} - {max(year_cols_pop)}")

In [ ]:
# Verification specifique : Paris 2024 est-il present ?
print("=== Verification Paris 2024 ===")

# Dans results.csv
paris_results = results[results["Games"].str.contains("2024", na=False)]
print(f"\nresults.csv -- participations 2024 : {len(paris_results):,}")
if len(paris_results) > 0:
    print(f"  Disciplines : {paris_results['Discipline'].nunique()}")
    print(f"  Medailles : {paris_results['Medal'].notna().sum()}")

# Dans olympics_1896_2024.csv
paris_olympics = olympics[olympics["event_year"] == 2024]
print(f"\nolympics_1896_2024.csv -- medailles 2024 : {len(paris_olympics):,}")
if len(paris_olympics) > 0:
    print(f"  Ville : {paris_olympics['city'].unique()}")
    print(f"  Disciplines : {paris_olympics['discipline'].nunique()}")

In [ ]:
# Nettoyage de la colonne temporaire
results.drop(columns=["year"], inplace=True)

### Coherence entre `olympics_1896_2024` et `city_locations`

Verifions que toutes les villes du dataset de medailles ont des coordonnees.

In [ ]:
olympics_cities = set(olympics["city"].dropna().unique())
ref_cities = set(city_locations["City"].dropna().unique())

missing_cities = olympics_cities - ref_cities

print(f"Villes dans olympics_1896_2024 : {len(olympics_cities)}")
print(f"Villes dans city_locations :     {len(ref_cities)}")
print(f"\nVilles sans coordonnees ({len(missing_cities)}) :")
if missing_cities:
    for city in sorted(missing_cities):
        print(f"  {city}")
else:
    print("  Aucune -- toutes les villes ont des coordonnees.")

---

## Conclusion

### Donnees acquises

Nous disposons de **7 datasets** couvrant l'ensemble de l'histoire olympique,
de 1896 (Athenes) a 2024 (Paris) :

| Dataset | Contenu | Volume |
|---------|---------|--------|
| `bios.csv` | Biographies des athletes | ~145 500 athletes |
| `bios_locs.csv` | Biographies + geolocalisation | ~145 500 athletes |
| `results.csv` | Participations et resultats | ~308 400 participations |
| `olympics_1896_2024.csv` | Medailles consolidees | ~24 000 medailles |
| `noc_regions.csv` | Mapping NOC -> pays | 229 codes |
| `populations.csv` | Population par pays (1960-2023) | 266 entites |
| `city_locations.csv` | Coordonnees des villes hotes | 55 editions |

### Points d'attention pour le nettoyage

Plusieurs chantiers se dessinent pour le prochain notebook :

1. **Valeurs manquantes** -- Les colonnes biographiques (`Died`, `Nick/petnames`, `Measurements`) ont des taux de NaN eleves. Pour les mensurations (`height_cm`, `weight_kg`), une strategie d'imputation ou de filtrage sera necessaire.

2. **Colonne parasite** -- `Unnamed: 7` dans `results.csv` est a supprimer.

3. **Harmonisation des codes pays** -- Les codes NOC (results) et ISO (populations) ne sont pas identiques. Une table de correspondance devra etre construite.

4. **Parsing des dates** -- La colonne `Born` de `bios.csv` est en texte libre ; `bios_locs.csv` offre une alternative deja structuree.

5. **Doublons potentiels** -- A verifier dans `results.csv` (un meme athlete pourrait apparaitre plusieurs fois pour la meme epreuve).

6. **Coherence des noms** -- Les noms d'athletes et de pays peuvent differer entre `results.csv` et `olympics_1896_2024.csv` (sources differentes).

### Sources completes

| Source | URL |
|--------|-----|
| KeithGalli/Olympics-Dataset | https://github.com/KeithGalli/Olympics-Dataset |
| KeithGalli/Olympics-Dataset (clean-data) | https://github.com/KeithGalli/Olympics-Dataset/tree/main/clean-data |
| DjangoMustang/Olympics-1896-2024-Tableau | https://github.com/DjangoMustang/Olympics-1896-2024-Tableau |
| Banque Mondiale (via KeithGalli) | https://data.worldbank.org/indicator/SP.POP.TOTL |